<img src="../_static/pymt-logo-header-text.png">

## Coastline Evolution Model + Waves

* Link to this notebook: https://github.com/csdms/pymt/blob/master/notebooks/cem_and_waves.ipynb
* Install command: `$ conda install notebook pymt_cem`

This example explores how to use a BMI implementation to couple the Waves component with the Coastline Evolution Model component.

### Links
* [CEM source code](https://github.com/csdms/cem-old): Look at the files that have *deltas* in their name.
* [CEM description on CSDMS](http://csdms.colorado.edu/wiki/Model_help:CEM): Detailed information on the CEM model.

### Interacting with the Coastline Evolution Model BMI using Python

Some magic that allows us to view images within the notebook.

In [35]:
%matplotlib inline
import numpy as np

Import the `Cem` class, and instantiate it. In Python, a model with a BMI will have no arguments for its constructor. Note that although the class has been instantiated, it's not yet ready to be run. We'll get to that later!

In [36]:
from pymt import models
from test_compose import cemtest

In [37]:
cem, waves = models.Cem(), models.Waves()

cem_compose = cemtest(waves,cem)

/opt/conda/lib/python3.10/site-packages/model_metadata/model_parameter.py:160: UserWarning: 3650.0: floating point number passed as an integer parameter, value will be truncated to 3650
  return IntParameter(value, **kwds)


Even though we can't run our waves model yet, we can still get some information about it. *Just don't try to run it.* Some things we can do with our model are get the names of the input variables.

In [38]:
waves.get_output_var_names()

('sea_surface_water_wave__min_of_increment_of_azimuth_angle_of_opposite_of_phase_velocity',
 'sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity',
 'sea_surface_water_wave__mean_of_increment_of_azimuth_angle_of_opposite_of_phase_velocity',
 'sea_surface_water_wave__max_of_increment_of_azimuth_angle_of_opposite_of_phase_velocity',
 'sea_surface_water_wave__height',
 'sea_surface_water_wave__period')

In [39]:
cem.get_input_var_names()

('sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity',
 'land_surface_water_sediment~bedload__mass_flow_rate',
 'sea_surface_water_wave__period',
 'sea_surface_water_wave__height',
 'land_surface__elevation',
 'model__time_step')

We can also get information about specific variables. Here we'll look at some info about wave direction. This is the main input of the Cem model. Notice that BMI components always use [CSDMS standard names](http://csdms.colorado.edu/wiki/CSDMS_Standard_Names). The CSDMS Standard Name for wave angle is,

    "sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"

Quite a mouthful, I know. With that name we can get information about that variable and the grid that it is on (it's actually not a one).

OK. We're finally ready to run the model. Well not quite. First we initialize the model with the BMI **initialize** method. Normally we would pass it a string that represents the name of an input file. For this example we'll pass **None**, which tells Cem to use some defaults.

In [40]:

bmi2list = {'number_of_rows' : 100, 'number_of_cols' : 200, 'grid_spacing' : 200.0}


args = cem_compose.setup(args1 = {}, args2 = bmi2list)

print(args)

cem_compose.initialize(args[1],args[2])

(<test_compose.cemtest.<locals>.ComposedBmi object at 0x7f2c92efc6a0>, ('waves.txt', '/tmp/tmp_vasuvu1'), ('cem.txt', '/tmp/tmpp3xeija5'))
Condition Initial 


Setting end time to 3650
CEM: trying to open file: cem.txt
CEM: line: 100, 200, 200.0, 1

CEM: number of rows, columns: 100, 200
*** Grid size is (0,0)
*** Requested size is (100,400)
*** New grid size is (100,400)
/opt/conda/lib/python3.10/site-packages/landlab/graph/graph.py:883: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  return self.ds.dims["patch"]
/opt/conda/lib/python3.10/site-packages/landlab/graph/graph.py:492: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  return self.ds.dims["link"]


<test_compose.cemtest.<locals>.ComposedBmi at 0x7f2c92efc6a0>

Here I define a convenience function for plotting the water depth and making it look pretty. You don't need to worry too much about it's internals for this tutorial. It just saves us some typing later on.

In [41]:
def plot_coast(spacing, z):
    import matplotlib.pyplot as plt

    xmin, xmax = 0.0, z.shape[1] * spacing[1] * 1e-3
    ymin, ymax = 0.0, z.shape[0] * spacing[0] * 1e-3

    plt.imshow(z, extent=[xmin, xmax, ymin, ymax], origin="lower", cmap="ocean")
    plt.colorbar().ax.set_ylabel("Water Depth (m)")
    plt.xlabel("Along shore (km)")
    plt.ylabel("Cross shore (km)")

It generates plots that look like this. We begin with a flat delta (green) and a linear coastline (y = 3 km). The bathymetry drops off linearly to the top of the domain.

In [42]:
grid_id = cem.var_grid("sea_water__depth")
spacing = cem.grid_spacing(grid_id)
shape = cem.grid_shape(grid_id)
z = np.empty(shape)
cem_compose.get_value("sea_water__depth", z)
plot_coast(spacing, z)

/home/Documents/University/bmi-compose/test/test_compose.py:56: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if output == '':


Allocate memory for the sediment discharge array and set the discharge at the coastal cell to some value.

In [43]:
qs = np.zeros_like(z)
qs[0, 100] = 750

The CSDMS Standard Name for this variable is:

    "land_surface_water_sediment~bedload__mass_flow_rate"

You can get an idea of the units based on the quantity part of the name. "mass_flow_rate" indicates mass per time. You can double-check this with the BMI method function **get_var_units**.

In [44]:
cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")

/tmp/ipykernel_70371/2188898298.py:1: DeprecationWarning: Call to deprecated method get_var_units. (use var_units)
  cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")


'kg / s'

In [45]:
waves.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_asymmetry_parameter",
    0.3,
)
waves.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_highness_parameter",
    0.7,
)
cem.set_value("sea_surface_water_wave__height", 2.0)
cem.set_value("sea_surface_water_wave__period", 7.0)

array([7.])

Set the bedload flux and run the model.

In [46]:

qs = np.zeros_like(z)
qs[0, 100] = 750

set_dict = {'sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity' :
            'sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity',
            'land_surface_water_sediment~bedload__mass_flow_rate' :
            qs
            }

for time in range(3000):
    cem_compose.update(set_dict)

    
cem_compose.get_value("sea_water__depth", z)


==== WaveAngle: -5.15:  MASS Percent: 0.0000:  Time Step: 0
==== WaveAngle: -71.99:  MASS Percent: 1.0029:  Time Step: 1000
==== WaveAngle: -51.19:  MASS Percent: 1.0057:  Time Step: 2000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [47]:
plot_coast(spacing, z)

Let's add another sediment source with a different flux and update the model.

In [48]:
qs[0, 150] = 500
for time in range(3750):
    cem_compose.update(set_dict)

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -46.33:  MASS Percent: 1.0084:  Time Step: 3000
==== WaveAngle: -50.89:  MASS Percent: 1.0132:  Time Step: 4000
==== WaveAngle: 79.63:  MASS Percent: 1.0179:  Time Step: 5000
==== WaveAngle: -29.41:  MASS Percent: 1.0223:  Time Step: 6000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [49]:
plot_coast(spacing, z)

Here we shut off the sediment supply completely.

In [50]:
qs.fill(0.0)
for time in range(4000):
    cem_compose.update(set_dict)

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -46.46:  MASS Percent: 1.0253:  Time Step: 7000
==== WaveAngle: -75.46:  MASS Percent: 1.0251:  Time Step: 8000
==== WaveAngle: 57.58:  MASS Percent: 1.0249:  Time Step: 9000
==== WaveAngle: -12.22:  MASS Percent: 1.0247:  Time Step: 10000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [51]:
plot_coast(spacing, z)